# 01. 물리 기반 기준 모델

**기준 모델(baseline)** 은 "AI가 이것보다 나아야 의미가 있다"는 비교 기준이에요. 이번에는 논문(Udofia et al., 2025)의 물리식 **M3**를 기준 모델로 써요.

$$F = A \cdot \dot\varepsilon^{\,n} \cdot w \cdot \ell$$

| 기호 | 뜻 | 계산 |
|---|---|---|
| $F$ | 하중 (N) | 예측 대상 |
| $w$ | 폭 (mm) | 데이터 |
| $\ell$ | 접촉 길이 (mm) | $\sqrt{r\,(h_o-h_f)}$ |
| $\varepsilon$ | 변형률 | $\ln(h_o / h_f)$ |
| $v$ | 출측 속도 (mm/s) | $1096 \times 27.7 / h_f$ (질량 보존, 아래 1번 참고) |
| $\dot\varepsilon$ | 변형 속도 (1/s) | $\varepsilon \cdot v / \ell$ |
| $A, n$ | 소재 계수 | 데이터로 맞춤. 논문 값은 $A=113.1$, $n=0.314$ |

뜻으로 풀면 **하중 = 소재가 버티는 힘(MPa) × 롤에 닿는 면적(폭 × 접촉 길이)** 이에요. 소재가 버티는 힘은 빨리 누를수록 커지므로 $A \cdot \dot\varepsilon^{\,n}$으로 나타내요.

### 비교하는 모델 4개

| 모델 | 설명 |
|---|---|
| 스탠드 평균 | 물리 계산 없이, 스탠드별 평균 하중을 그대로 예측값으로 씀. 가장 단순한 비교 기준 |
| M3 논문 계수 | 논문의 $A=113.1$, $n=0.314$를 그대로 씀 (학습 없음) |
| M3 재학습 | 같은 식, $A$와 $n$을 우리 학습 데이터로 다시 맞춤 (계수 2개) |
| M3 강종별 | 강종마다 $A$와 $n$을 따로 맞춤 (계수 20개, 논문 Table 6과 같은 방식) |

### 평가 방법: 코일 단위 5-fold
- 코일을 5묶음으로 나누고, 4묶음으로 계수를 맞춘 뒤 남은 1묶음을 예측해요. 이걸 5번 돌려 모든 행이 한 번씩 "처음 보는 코일"로 예측돼요.
- 같은 코일의 F1~F5가 학습과 평가에 나뉘어 들어가지 않게 해서, 처음 보는 코일에 대한 실력을 재요.
- **다음 단계 AI 모델도 똑같은 5묶음으로 평가해서** 숫자를 바로 비교할 수 있게 해요.

## 0. 준비

**데이터:** 팀 공통 정리 기준인 `pjh/data_utils.py`의 `load_clean()`으로 불러와요. 스탠드 누락·두께 불연속·폭 변동 코일을 뺀 코일 7,533개예요. 열 이름과 파생 변수는 kdh 함수(`kdh/src/prepare_data.py`)로 맞춰요.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager
from sklearn.model_selection import GroupKFold
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

# 한글 글꼴: 맥(AppleGothic), 윈도우(Malgun Gothic), 리눅스(NanumGothic) 중 있는 것을 씀
installed = {f.name for f in font_manager.fontManager.ttflist}
for font in ["AppleGothic", "Malgun Gothic", "NanumGothic"]:
    if font in installed:
        plt.rcParams["font.family"] = font
        break
plt.rcParams["axes.unicode_minus"] = False

BLUE, BLUE_DARK = "#2a78d6", "#184f95"
plt.rcParams.update({
    "figure.dpi": 110,
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "axes.titlesize": 12,
    "xtick.color": "#52514e", "ytick.color": "#52514e",
    "axes.grid": True, "axes.axisbelow": True, "grid.color": "#e1e0d9", "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False,
})

STANDS = ["F1", "F2", "F3", "F4", "F5"]

# 저장소 맨 위 폴더 찾기 (노트북을 ojs/에서 열든 저장소 맨 위에서 열든 동작)
REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pjh/data_utils.py").exists())
sys.dont_write_bytecode = True   # 다른 팀원 폴더에 캐시(__pycache__)를 만들지 않음
sys.path.insert(0, str(REPO / "pjh"))
sys.path.insert(0, str(REPO / "kdh/src"))
from data_utils import load_clean                     # pjh 데이터 정리 기준 (팀 공통)
from prepare_data import COLS, add_features           # kdh 열 이름·파생 변수

clean, clean_info = load_clean()                      # 스탠드 누락·두께 불연속·폭 변동 코일 제외
df = add_features(clean.rename(columns=COLS))
df["chain_ok"] = True                                 # 정리 후 남은 코일은 모두 연속
print(f"{len(df):,}행, 코일 {df['coil'].nunique():,}개, 제외 코일 {clean_info['제외 코일(중복 제거)']}개")

## 1. 물리 변수 계산

논문은 압연 속도를 측정하지 않았어요. 대신 "모든 코일이 F1에 평균 1,096mm/s로 들어간다"고 가정하고, 질량 보존(두께가 얇아진 만큼 빨라짐)으로 스탠드마다 속도를 계산했어요. 우리도 똑같이 해요.

> **F1 입측 두께 기준값은 25.8mm가 아니라 27.7mm를 써요.**
> - 논문 본문에는 25.8mm라고 적혀 있어요. 그런데 이 값으로 계산하면 논문 계수가 하중을 평균 0.42 MN 낮게 예측해요.
> - 우리 데이터의 실제 F1 입측 두께 평균인 27.7mm로 계산하면, 우리가 다시 맞춘 $A$가 113.0으로 논문 값(113.1)과 같아지고 치우침도 사라져요.
> - 그래서 논문은 실제로 27.7mm로 계산했고, 본문의 25.8mm는 데이터와 맞지 않는 Table 3 값을 옮겨 적은 것으로 봐요.
> - 이 값은 모든 행에 똑같이 곱해지는 상수라, 계수를 다시 맞추는 모델(재학습, 강종별)의 성능에는 영향이 없어요.

In [ ]:
V1 = 1096.0   # 논문 가정값: F1 입측 평균 속도 (mm/s)
HO1 = 27.7    # F1 입측 평균 두께 (mm), 우리 데이터 평균 (위 설명 참고)

df["strain"] = np.log(df["h_in"] / df["h_out"])          # 변형률
df["v"] = V1 * HO1 / df["h_out"]                         # 출측 속도 (mm/s)
df["strain_rate"] = df["strain"] * df["v"] / df["L"]     # 변형 속도 (1/s)
df["area"] = df["W"] * df["L"]                           # 롤에 닿는 면적 = 폭 × 접촉 길이 (mm²)

view = df.groupby("stand")[["strain", "v", "L", "strain_rate"]].mean().round(2)
view.columns = ["변형률", "속도 (mm/s)", "접촉 길이 (mm)", "변형 속도 (1/s)"]
display(view)

뒤 스탠드로 갈수록 판이 얇아져 빨라지고(속도), 롤에 닿는 길이는 짧아져서 **변형 속도가 F1의 약 7배**까지 커져요. 그래서 F5는 압하율이 작아도 소재가 더 단단하게 버텨요.

## 2. 코일 단위 5-fold 나누기

In [ ]:
gkf = GroupKFold(n_splits=5)
df["fold"] = -1
for k, (_, test_idx) in enumerate(gkf.split(df, groups=df["coil"])):
    df.loc[df.index[test_idx], "fold"] = k

fold_view = df.groupby("fold").agg(코일_수=("coil", "nunique"), 행_수=("coil", "size"))
display(fold_view)

# 같은 코일이 두 묶음에 걸치지 않는지 확인
assert df.groupby("coil")["fold"].nunique().max() == 1

## 3. 모델 만들기

M3 식의 양변에 로그를 씌우면 직선이 돼요. 그래서 직선 맞추기(`np.polyfit`)로 $A$와 $n$을 구할 수 있어요.

$$\ln\frac{F}{w \cdot \ell} = \ln A + n \cdot \ln\dot\varepsilon$$

In [ ]:
def fit_m3(d):
    """학습 데이터 d로 M3의 A, n을 구함 (로그 변환 후 직선 맞추기)"""
    y = np.log(d["F_N"] / d["area"])
    x = np.log(d["strain_rate"])
    n, lnA = np.polyfit(x, y, 1)
    return np.exp(lnA), n


def predict_m3(d, A, n):
    """M3 식으로 하중(MN)을 계산"""
    return A * d["strain_rate"] ** n * d["area"] / 1e6


MODELS = ["스탠드 평균", "M3 논문 계수", "M3 재학습", "M3 강종별"]
pred = pd.DataFrame(index=df.index, columns=MODELS, dtype=float)
pred["M3 논문 계수"] = predict_m3(df, 113.1, 0.314)   # 학습 없이 논문 계수 그대로

coef_rows = []
for k in range(5):
    train, test = df[df["fold"] != k], df[df["fold"] == k]

    pred.loc[test.index, "스탠드 평균"] = test["stand"].map(train.groupby("stand")["F_MN"].mean())

    A, n = fit_m3(train)
    pred.loc[test.index, "M3 재학습"] = predict_m3(test, A, n)
    coef_rows.append({"fold": k, "grade": "전체", "A": A, "n": n})

    for grade, train_g in train.groupby("grade"):
        A_g, n_g = fit_m3(train_g)
        idx = test.index[test["grade"] == grade]
        pred.loc[idx, "M3 강종별"] = predict_m3(test.loc[idx], A_g, n_g)
        coef_rows.append({"fold": k, "grade": grade, "A": A_g, "n": n_g})

assert pred.notna().all().all()
print("모든 행 예측 완료")

## 4. 성능 비교

- **R²**: 1에 가까울수록 좋아요. 하중의 들쭉날쭉함을 얼마나 설명하는지예요.
- **MAE**: 평균적으로 몇 MN 틀리는지. 작을수록 좋아요.
- **RMSE**: MAE와 비슷하지만 크게 틀린 것에 벌점을 더 줘요.
- **평균 오차율**: 실제 하중 대비 몇 % 틀리는지.
- **치우침**: 예측 − 실제의 평균. 0보다 작으면 전체적으로 낮게 예측해요.

In [ ]:
y = df["F_MN"]

def scores(p):
    return pd.Series({
        "R²": r2_score(y, p),
        "MAE (MN)": mean_absolute_error(y, p),
        "RMSE (MN)": root_mean_squared_error(y, p),
        "평균 오차율 (%)": ((p - y).abs() / y).mean() * 100,
        "치우침 (MN)": (p - y).mean(),
    })

result = pd.DataFrame({m: scores(pred[m]) for m in MODELS}).T
display(result.style.format({"R²": "{:.3f}", "MAE (MN)": "{:.2f}", "RMSE (MN)": "{:.2f}",
                             "평균 오차율 (%)": "{:.1f}", "치우침 (MN)": "{:+.2f}"}))

- **물리식 하나로 R² 0.944**예요. 스탠드 평균(R² 0.731)보다 훨씬 좋아서, 물리식이 하중의 구조를 잘 잡는다는 뜻이에요.
- **기준선은 논문 모델과 같은 M3 재학습이에요: R² 0.944, MAE 1.27 MN, 평균 오차율 7.4%.** 논문의 0.946과 숫자가 다른 이유는 바로 아래 칸에서 확인해요. 모델은 같고 R²를 계산하는 방식만 달라요.
- M3 논문 계수(학습 없음)도 거의 같아요. 다만 논문이 이 데이터로 맞춘 값이라 처음 보는 코일 평가가 아니어서, 기준선은 fold마다 다시 맞춘 M3 재학습으로 잡아요.
- 참고로 M3 강종별(계수 20개)은 R² 0.954, MAE 1.05 MN이에요. 물리식을 강종마다 따로 맞추면 여기까지 된다는 참고 기준이에요.

### 논문의 R² 0.946과 비교하기

논문은 두 가지가 달라요.
1. 계수를 **전체 데이터**로 맞추고 **같은 데이터**로 평가했어요(처음 보는 코일로 평가하지 않음).
2. 식을 로그로 바꿔 직선 맞추기를 했기 때문에, R²도 하중이 아니라 **로그 하중(ln F)** 으로 계산했어요.

같은 방식으로 계산해 보면:

In [ ]:
A_all, n_all = fit_m3(df)
p_all = predict_m3(df, A_all, n_all)
print(f"전체 데이터로 맞춘 계수: A = {A_all:.1f}, n = {n_all:.3f}   (논문 113.1, 0.314)")
print(f"[논문 방식] 로그 하중 기준 R²     : {r2_score(np.log(y), np.log(p_all)):.4f}   ← 논문 0.946")
print(f"같은 계수, 하중(MN) 기준 R²       : {r2_score(y, p_all):.3f}")
print(f"[이 노트북] 코일 5-fold, 하중 기준 R²: {r2_score(y, pred['M3 재학습']):.3f}")

- 논문 방식(전체 데이터로 맞추고 로그 하중으로 R²)으로 계산하면 0.9465로 **논문의 0.946이 그대로 나와요.**
- 같은 계수를 하중(MN) 기준으로 계산하면 0.944예요. 로그로 바꾸면 오차를 "몇 MN"이 아니라 "몇 %"에 가깝게 보게 돼서 값이 조금 달라져요.
- 처음 보는 코일로 평가해도(코일 단위 5-fold) 0.944로 같아요. 계수가 2개뿐이라 외워서 맞히는 일(과적합)이 거의 없기 때문이에요. 논문의 80/20 검증(Table 4)도 test R² 0.945, RMSE 1.65 MN으로 우리(RMSE 1.67 MN)와 거의 같아요.
- **AI 모델은 하중(MN) 기준으로 평가하므로, 비교 기준은 0.944예요.** 0.946과 0.944는 같은 모델을 다른 자로 잰 값이에요.

## 5. 우리가 구한 계수 vs 논문 계수

5-fold에서 구한 $A$, $n$의 평균을 논문 값과 나란히 놓았어요. 논문 Table 6은 강종별 값이에요.

In [ ]:
paper = pd.DataFrame({
    "grade": ["전체"] + [f"Grade{i:02d}" for i in range(1, 11)],
    "논문 A": [113.1, 117.6, 103.6, 122.6, 108.2, 94.9, 102.2, 123.1, 130.3, 103.8, 102.6],
    "논문 n": [0.314, 0.291, 0.346, 0.288, 0.342, 0.362, 0.364, 0.285, 0.295, 0.344, 0.353],
}).set_index("grade")

ours = pd.DataFrame(coef_rows).groupby("grade")[["A", "n"]].mean()
ours.columns = ["우리 A", "우리 n"]
coef_table = ours.join(paper).loc[paper.index][["우리 A", "논문 A", "우리 n", "논문 n"]]
display(coef_table.style.format({"우리 A": "{:.1f}", "논문 A": "{:.1f}", "우리 n": "{:.3f}", "논문 n": "{:.3f}"}))

- 코일 단위 5-fold로 구한 $A$, $n$이 논문 Table 6과 강종마다 거의 같아요(대부분 $A$ 차이 0.5 이하, $n$ 차이 0.001 이하). 정리 기준에서 코일을 뺀 Grade05(A 93.4 vs 94.9)와 전체(A 114.1 vs 113.1)만 조금 달라요.
- → 우리 데이터와 계산 방식이 논문과 같다는 확인이에요. 발표에서 "논문의 물리식을 그대로 재현했다"고 말할 수 있어요.
- $n$은 강종에 따라 0.285~0.364예요. 빨리 누를 때 더 단단해지는 정도가 강종마다 다르다는 뜻이에요.

## 6. 스탠드별·강종별 오차

평균 오차율(%)로 비교해요. F5는 하중 자체가 작아서 MN으로 보면 오차가 작아 보이기 때문이에요.

In [ ]:
err_pct = pred.sub(y, axis=0).abs().div(y, axis=0) * 100

by_stand = err_pct.groupby(df["stand"]).mean().round(1)
by_stand["평균 하중 (MN)"] = y.groupby(df["stand"]).mean().round(1)
print("스탠드별 평균 오차율 (%)")
display(by_stand)

by_grade = err_pct.groupby(df["grade"]).mean().round(1)
by_grade["코일 수"] = df.groupby("grade")["coil"].nunique()
print("강종별 평균 오차율 (%)")
display(by_grade)

- **스탠드:** 공통 계수(논문·재학습)는 F5에서 오차율이 11%로 커요. 강종별로 맞추면 6.6%로 줄어요. 뒤 스탠드일수록 강종에 따른 단단함 차이가 커져서, 계수 하나로는 F5를 잘 맞추지 못해요. 강종별 모델은 F3·F4가 가장 잘 맞고(4.4~4.9%), F1·F2·F5는 6.3~6.6%예요.
- **강종:** Grade09·Grade10이 7.5% 이상으로 가장 커요. 논문에서도 이 두 강종의 R²가 가장 낮았어요(0.907, 0.912).

## 7. 그래프로 보기

- 왼쪽: 실제 하중(가로)과 기준선 M3 재학습 예측(세로). 점이 대각선에 붙을수록 잘 맞아요.
- 오른쪽: 스탠드별로 **예측 − 실제**의 분포. 상자가 0선보다 위면 그 스탠드를 높게, 아래면 낮게 예측하는 버릇이 있다는 뜻이에요.

In [ ]:
best = "M3 재학습"   # 기준선 (논문 모델)
resid = pred[best] - y

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6))
ax1.scatter(y, pred[best], s=3, alpha=0.15, color=BLUE, linewidths=0)
lim = [0, max(y.max(), pred[best].max()) * 1.03]
ax1.plot(lim, lim, color="#52514e", linewidth=1, linestyle="--")
ax1.set_xlim(lim); ax1.set_ylim(lim)
ax1.set_aspect("equal")
ax1.set_xlabel("실제 하중 (MN)")
ax1.set_ylabel("예측 하중 (MN)")
ax1.set_title(f"{best}: 실제 vs 예측 (R² {r2_score(y, pred[best]):.3f})")

ax2.boxplot(
    [resid[df["stand"] == s] for s in STANDS], tick_labels=STANDS, widths=0.5,
    whis=(5, 95), showfliers=False, patch_artist=True,
    boxprops=dict(facecolor="#cde2fb", edgecolor=BLUE), medianprops=dict(color=BLUE_DARK, linewidth=2),
    whiskerprops=dict(color=BLUE), capprops=dict(color=BLUE),
)
ax2.axhline(0, color="#52514e", linewidth=1)
ax2.xaxis.grid(False)
ax2.set_ylabel("예측 - 실제 (MN)")
ax2.set_title(f"{best}: 스탠드별 오차 (수염 5%~95%)")
plt.tight_layout()
plt.show()

print("스탠드별 치우침 (예측 − 실제의 평균, MN):")
print(resid.groupby(df["stand"]).mean().round(2).to_string())

- 점이 대각선 주변에 잘 모여 있지만, 25 MN 이상(주로 F1·F2)에서 넓게 퍼져요.
- 스탠드별 버릇이 있어요. F1은 평균 0.29 MN 낮게, F2는 0.44 MN 높게 예측해요. F1·F2는 오차 폭(수염)도 가장 넓어요.
- → 물리식이 놓치는 스탠드별·조건별 패턴이 남아 있다는 뜻이에요. 다음 단계에서 AI 모델(GBDT, MLP)과 하이브리드(물리식 × AI 보정)가 이 부분을 잡는지 봐요.

## 정리: AI 모델이 넘어야 할 기준

| 항목 | 내용 |
|---|---|
| 기준선 | **M3 (논문 모델): R² 0.944, MAE 1.27 MN, 평균 오차율 7.4%.** 논문의 0.946은 같은 모델의 로그 하중 기준 R²예요 |
| 참고 기준 | M3 강종별: R² 0.954, MAE 1.05 MN, 평균 오차율 5.8% |
| 평가 방법 | 코일 단위 5-fold (`GroupKFold(n_splits=5)`, 그룹 = 코일 번호). AI 모델도 같은 코드로 나눠서 비교해요 |
| 물리 변수 | 속도 = 1096 × 27.7 / 출측 두께, 변형 속도 = 변형률 × 속도 / 접촉 길이. AI 모델 입력으로도 쓸 수 있어요 |
| 남은 오차 | F1·F2의 큰 하중 구간, 스탠드별 치우침(F1 −0.29, F2 +0.44 MN), Grade09·10 |
| 최적화와의 관계 | M3 식은 한 스탠드에서 더 누르면 그 스탠드 하중이 반드시 커지는 구조라 최적화에 그대로 쓸 수 있어요. AI 모델은 이 성질을 따로 확인해야 해요 |